# Module 5 — Outlet Behavioural Profiling

**Purpose:** Engineer a clean, statistically validated outlet feature matrix for K-means clustering. Features must be scaled, non-collinear, and free of degenerate distributions. The output of this module is the direct input to Module 6.

**Inputs:** `outlet_monthly_activity.parquet`, `outlet_features.parquet` (from Module 0)

---

## Analysis Roadmap

1. **Normality Testing** — Shapiro-Wilk or D'Agostino-Pearson on each numeric feature
2. **Outlier Detection** — IQR fence at 3×, flagging structural outliers
3. **Transformation** — Log-transform right-skewed features, Winsorise extreme ratios
4. **Multicollinearity** — Pearson correlation matrix, VIF calculation
5. **Final Scaling** — Apply StandardScaler
#

In [2]:
import sys
from pathlib import Path
#

In [3]:
PROJECT_ROOT = Path("../../..")
sys.path.insert(0, str(PROJECT_ROOT))
#

In [4]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
from scipy import stats
import os
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

warnings.filterwarnings('ignore')

# Set visualization defaults
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

try:
    # Import McKinsey theme
    from src.viz.mckinsey_theme import (
        MCKINSEY_RC, BG_DARK, TEXT_WHITE, TEXT_DARK, GRID_COLOR,
        LINE_NOMINAL, LINE_VOLUME, LINE_REAL, HIGHLIGHT,
        create_mckinsey_figure, add_end_label
    )
    # Apply McKinsey theme
    matplotlib.rcParams.update(MCKINSEY_RC)
except ImportError:
    print("Warning: McKinsey theme not found. Proceeding with default styles.")

print("Libraries imported successfully.")
#

Libraries imported successfully.


## Section 1: Load Analytical Datasets

Load the pre-processed analytical datasets from Module 0.
#

In [6]:
# Define data paths
intermediate_path = "../../../data/Intermediate"

# Load analytical datasets
print("Loading analytical datasets from Module 0...")

outlet_features = pd.read_parquet(os.path.join(intermediate_path, "outlet_features.parquet"))
outlet_monthly_activity = pd.read_parquet(os.path.join(intermediate_path, "outlet_monthly_activity.parquet"))

print(f"✓ outlet_features: {outlet_features.shape}")
print(f"  Columns: {list(outlet_features.columns)[:5]}...")
print(f"✓ outlet_monthly_activity: {outlet_monthly_activity.shape}")
print(f"  Columns: {list(outlet_monthly_activity.columns)[:5]}...")

# Filter active outlets for clustering
active_outlets = outlet_features[outlet_features['is_active'] == True].copy()
churned_outlets = outlet_features[outlet_features['is_active'] == False].copy()

print(f"Total active outlets: {len(active_outlets)}")
print(f"Total churned outlets: {len(churned_outlets)}")
#

Loading analytical datasets from Module 0...
✓ outlet_features: (116183, 18)
  Columns: ['outlet_id', 'total_net_sales', 'total_quantity', 'active_months', 'loyalty_ratio']...
✓ outlet_monthly_activity: (7319529, 15)
  Columns: ['outlet_id', 'year_month', 'n_invoices', 'gross_quantity', 'gross_sales']...
Total active outlets: 66736
Total churned outlets: 49447


## Section 2: Normality Testing

Apply D'Agostino-Pearson test on each numeric feature to check for normality.
#

In [8]:
numeric_features = [
    'total_net_sales', 'total_quantity', 'active_months', 'loyalty_ratio',
    'avg_txn_per_active_month', 'avg_net_sales_per_txn', 'avg_qty_per_txn',
    'n_unique_skus', 'n_unique_categories', 'return_rate', 'recency_months',
    'momentum', 'pct_cash_txn'
]

print("=" * 80)
print("NORMALITY TESTING (D'Agostino-Pearson)")
print("=" * 80)

normality_results = []
for feature in numeric_features:
    # Dropna to avoid test failure on nulls
    data = active_outlets[feature].dropna()
    stat, p = stats.normaltest(data)
    is_normal = p > 0.05
    skewness = stats.skew(data)
    
    normality_results.append({
        'Feature': feature,
        'Statistic': stat,
        'p-value': p,
        'Skewness': skewness,
        'Is Normal': is_normal
    })

normality_df = pd.DataFrame(normality_results)
print(normality_df)
print("Note: Most features will fail normality due to business skew. This justifies log transformation.")
#

NORMALITY TESTING (D'Agostino-Pearson)
                     Feature      Statistic  p-value   Skewness  Is Normal
0            total_net_sales  190240.973891      0.0  38.052681      False
1             total_quantity  198441.453769      0.0  42.739448      False
2              active_months   12537.703828      0.0   0.438235      False
3              loyalty_ratio   12537.703828      0.0   0.438235      False
4   avg_txn_per_active_month   85307.038926      0.0   6.360494      False
5      avg_net_sales_per_txn  157118.431109      0.0  23.619483      False
6            avg_qty_per_txn  158129.557825      0.0  23.979640      False
7              n_unique_skus   25387.722242      0.0   1.893502      False
8        n_unique_categories    5712.987063      0.0   0.394611      False
9                return_rate   10254.853989      0.0   1.170286      False
10            recency_months            NaN      NaN        NaN      False
11                  momentum  224094.756362      0.0  60.1744

## Section 3: Outlier Detection

Apply IQR fence at 3× to detect structural outliers. Outlets breaching 3+ features are excluded from clustering but retained for separate analysis.
#

In [10]:
def detect_outliers_iqr(df, features, multiplier=3.0):
    outlier_flags = pd.DataFrame(index=df.index)
    
    for feature in features:
        Q1 = df[feature].quantile(0.25)
        Q3 = df[feature].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR
        
        # Flag outliers
        outlier_flags[f"{feature}_outlier"] = (df[feature] < lower_bound) | (df[feature] > upper_bound)
        
    outlier_flags['outlier_count'] = outlier_flags.sum(axis=1)
    return outlier_flags

outlier_flags = detect_outliers_iqr(active_outlets, numeric_features)

active_outlets['outlier_count'] = outlier_flags['outlier_count']
structural_outliers = active_outlets[active_outlets['outlier_count'] >= 3]
clustering_population = active_outlets[active_outlets['outlier_count'] < 3].copy()

print("=" * 80)
print("OUTLIER DETECTION SUMMARY")
print("=" * 80)
print(f"Total active outlets evaluated: {len(active_outlets)}")
print(f"Structural outliers detected (≥ 3 flags): {len(structural_outliers)} ({(len(structural_outliers)/len(active_outlets))*100:.2f}%)")
print(f"Population retained for clustering: {len(clustering_population)}")

# Revenue share of outliers
outlier_rev = structural_outliers['total_net_sales'].sum()
total_rev = active_outlets['total_net_sales'].sum()
print(f"Revenue share of outliers: {(outlier_rev / total_rev) * 100:.2f}%")
#

OUTLIER DETECTION SUMMARY
Total active outlets evaluated: 66736
Structural outliers detected (≥ 3 flags): 4534 (6.79%)
Population retained for clustering: 62202
Revenue share of outliers: 72.22%


## Section 4: Feature Transformation

Apply transformations to correct skewness and extreme values:
- Log-transform right-skewed features
- Winsorise momentum
- Bounded features remain unchanged
#

In [11]:
features_to_log = ['total_net_sales', 'total_quantity', 'avg_net_sales_per_txn', 'avg_qty_per_txn']

transformed_population = clustering_population.copy()

print("=" * 80)
print("FEATURE TRANSFORMATION")
print("=" * 80)

# 1. Log Transformations
for feature in features_to_log:
    transformed_col = f"{feature}_log"
    # Adding 1 to avoid log(0)
    transformed_population[transformed_col] = np.log1p(transformed_population[feature].clip(lower=0))
    
    old_skew = stats.skew(transformed_population[feature].dropna())
    new_skew = stats.skew(transformed_population[transformed_col].dropna())
    print(f"Log transformed {feature}: Skewness {old_skew:.2f} -> {new_skew:.2f}")

# 2. Winsorise momentum
lower_limit = transformed_population['momentum'].quantile(0.01)
upper_limit = transformed_population['momentum'].quantile(0.99)
transformed_population['momentum_winsor'] = transformed_population['momentum'].clip(lower_limit, upper_limit)
old_skew = stats.skew(transformed_population['momentum'].dropna())
new_skew = stats.skew(transformed_population['momentum_winsor'].dropna())
print(f"Winsorised momentum: Skewness {old_skew:.2f} -> {new_skew:.2f}")

# Select the final transformed feature set for clustering
transformed_features = [
    'total_net_sales_log', 'total_quantity_log', 'active_months', 'loyalty_ratio',
    'avg_txn_per_active_month', 'avg_net_sales_per_txn_log', 'avg_qty_per_txn_log',
    'n_unique_skus', 'n_unique_categories', 'return_rate', 'recency_months',
    'momentum_winsor', 'pct_cash_txn'
]

transformed_population = transformed_population[['outlet_id', 'territory_id', 'region', 'province'] + transformed_features].dropna()
#

FEATURE TRANSFORMATION
Log transformed total_net_sales: Skewness 5.00 -> -0.34
Log transformed total_quantity: Skewness 3.24 -> -0.49
Log transformed avg_net_sales_per_txn: Skewness 9.30 -> 0.21
Log transformed avg_qty_per_txn: Skewness 5.87 -> -0.05
Winsorised momentum: Skewness 49.69 -> 4.60


## Section 5: Multicollinearity (VIF & Correlation)

Assess feature correlations and compute Variance Inflation Factor (VIF). Drop features with VIF >= 5.
#

In [13]:
print("=" * 80)
print("MULTICOLLINEARITY & VIF")
print("=" * 80)

def calculate_vif(df, features):
    X = df[features].copy()
    # Add constant for VIF calculation
    X['intercept'] = 1
    vif_data = pd.DataFrame()
    vif_data["Feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data[vif_data['Feature'] != 'intercept']

current_features = transformed_features.copy()
dropped_features = []

# Iteratively remove features with VIF > 5
while True:
    vif_df = calculate_vif(transformed_population, current_features)
    max_vif = vif_df['VIF'].max()
    if max_vif > 5.0:
        feature_to_drop = vif_df.loc[vif_df['VIF'].idxmax(), 'Feature']
        print(f"Dropping {feature_to_drop} with VIF: {max_vif:.2f}")
        current_features.remove(feature_to_drop)
        dropped_features.append(feature_to_drop)
    else:
        break

print("Final VIF values:")
print(vif_df)

print(f"Final retained features ({len(current_features)}): {current_features}")
#

MULTICOLLINEARITY & VIF
Dropping active_months with VIF: inf
Dropping total_net_sales_log with VIF: 14629.79
Dropping total_quantity_log with VIF: 35.44
Dropping loyalty_ratio with VIF: 6.46
Dropping n_unique_skus with VIF: 5.07
Final VIF values:
                     Feature       VIF
0   avg_txn_per_active_month  2.868492
1  avg_net_sales_per_txn_log  3.154569
2        avg_qty_per_txn_log  2.668333
3        n_unique_categories  3.040254
4                return_rate  1.152957
5             recency_months       NaN
6            momentum_winsor  1.003674
7               pct_cash_txn  1.462724
Final retained features (8): ['avg_txn_per_active_month', 'avg_net_sales_per_txn_log', 'avg_qty_per_txn_log', 'n_unique_categories', 'return_rate', 'recency_months', 'momentum_winsor', 'pct_cash_txn']


## Section 6: Final Scaling & Export

Apply `StandardScaler` to retained features and export the scaled matrix for Module 6.
#

In [15]:
print("=" * 80)
print("FINAL SCALING")
print("=" * 80)

scaler = StandardScaler()
scaled_array = scaler.fit_transform(transformed_population[current_features])

scaled_df = pd.DataFrame(scaled_array, columns=current_features, index=transformed_population.index)
scaled_df['outlet_id'] = transformed_population['outlet_id']
scaled_df['territory_id'] = transformed_population['territory_id']
scaled_df['region'] = transformed_population['region']

print("Scaled features head:")
print(scaled_df[current_features].head())

# Verification
means = scaled_df[current_features].mean()
stds = scaled_df[current_features].std()

print("Verification - Means (should be ~0):")
print(means.round(4))
print("Verification - Stds (should be ~1):")
print(stds.round(4))

# Export
output_dir = os.path.join(intermediate_path, "clustering")
os.makedirs(output_dir, exist_ok=True)

scaled_df.to_parquet(os.path.join(output_dir, "outlet_features_scaled.parquet"))
print(f"Saved scaled feature matrix to: {os.path.join(output_dir, 'outlet_features_scaled.parquet')}")
#

FINAL SCALING
Scaled features head:
   avg_txn_per_active_month  avg_net_sales_per_txn_log  avg_qty_per_txn_log  \
2                  1.722860                   0.882542             0.365012   
4                 -0.216497                   0.728600             1.462479   
7                 -0.493548                   1.611213             1.017967   
8                 -1.047650                  -1.346379            -1.397444   
9                  1.168758                   2.912182             2.304148   

   n_unique_categories  return_rate  recency_months  momentum_winsor  \
2             0.071657    -0.571558             0.0        -0.419513   
4            -0.611436    -0.571558             0.0        -0.026721   
7            -1.294529    -0.571558             0.0        -0.419513   
8            -1.294529    -0.571558             0.0        -0.419513   
9            -0.611436    -0.571558             0.0        -0.419513   

   pct_cash_txn  
2     -1.386438  
4     -0.700028  
7 

---

## Executive Synthesis — What the Numbers Mean for the Business

> *Written for the CEO. Every figure is drawn directly from the data analysed above.*

---

### 1. How Big Is Your Active Outlet Base — and How Many Have You Lost?

Your commercial network spans **116,183 outlets** in total. Of those, 66,736 (57.4%) are currently active — meaning they have transacted within the measurement window. The remaining 49,447 outlets (42.6%) have churned: they were once customers but are no longer buying. Separately, your active outlets generate 7,319,529 monthly activity records, which is the raw material this analysis uses to characterise how each outlet behaves.

The active/churned split matters because it tells you the size of the recovery prize. Nearly half your known outlet universe has gone quiet. Even recapturing a fraction of those churned outlets — at the average revenue of an active outlet — represents a significant untapped upside that does not require winning a single new customer.

**So what:** You have as many lost outlets as active ones; a structured win-back programme targeting the churned 49,447 should be on your commercial agenda alongside new customer acquisition.

---

### 2. The Outlier Paradox — A Tiny Group of Outlets Owns Most of Your Revenue

Of the 66,736 active outlets, a statistical screening (using 3× the interquartile range across 13 behavioural dimensions) identified **4,534 outlets as structural outliers** — just 6.79% of the active base. These are not errors in the data; they are genuine extreme performers: outlets that buy far more frequently, in far larger volumes, across far more categories than the typical outlet.

The critical insight is their revenue weight: **these 4,534 outlets account for 72.22% of all active-outlet revenue**. The remaining 62,202 outlets — 93.21% of your active base — share just 27.78% of revenue. In practical terms, your top ~4,500 accounts are an entirely different commercial category from the rest of your network.

**So what:** Your commercial risk is heavily concentrated in a small group of power accounts; losing even 5–10% of those 4,534 outlets would have a more severe revenue impact than losing every one of your bottom 30,000 outlets combined.

---

### 3. Why Behavioural Data Cannot Be Taken at Face Value (Skewness & Transformation)

Every one of the 13 outlet behavioural features tested failed the normality test (D'Agostino-Pearson, p = 0.0 in all cases). This is not a data problem — it is a business reality: a small number of very high-activity outlets pulls every distribution sharply to the right. The most extreme case is momentum (a measure of recent sales acceleration), which had a raw skewness of 49.69 — meaning the distribution has an enormous tail driven by a handful of very high-growth outlets.

To make the data usable for segmentation, four features were log-transformed (reducing skewness from as high as 9.30 to near-zero) and the momentum feature was winsorised (capped at the 1st–99th percentile, reducing skewness from 49.69 to 4.60). These are standard statistical corrections, not alterations to the underlying business reality.

**So what:** The raw numbers in your CRM or ERP system for average outlets are systematically distorted by power accounts; any average-based target-setting or performance benchmarking you do today likely overstates what a typical outlet achieves.

---

### 4. Which Eight Signals Actually Differentiate Outlets from One Another (Feature Selection)

Starting from 13 candidate features, the analysis removed 5 that were redundant — i.e., they told the same story as other features already in the set (measured by Variance Inflation Factor, where VIF > 5 signals redundancy). The features dropped were: tenure (active_months, VIF = infinite — fully determined by other features), total sales volume (collinear), total quantity (collinear), loyalty ratio (VIF = 6.46), and SKU breadth (VIF = 5.07).

The **8 independent signals** that remain — each telling a distinct story about an outlet — are:
1. **Transaction frequency** (how often they order per active month)
2. **Sales value per transaction** (how much they spend per order, log-scaled)
3. **Basket size** (units per transaction, log-scaled)
4. **Category breadth** (how many product categories they buy across)
5. **Return rate** (what share of purchased volume comes back)
6. **Recency** (how recently they last transacted)
7. **Momentum** (whether their spend is accelerating or decelerating)
8. **Payment method mix** (share of cash vs. non-cash transactions)

**So what:** These 8 dimensions are the ones your field sales team should use to characterise and compare outlets — they are statistically independent, meaning each one adds genuinely new information about how an outlet behaves.

---

### 5. A Data Quality Flag: One Feature Is Carrying No Signal

The recency feature (months since last transaction) returned a standard deviation of **0.0** after scaling — meaning every one of the 62,202 retained active outlets has an identical recency value. This is a structural consequence of how "active" was defined: all outlets in this population transacted recently enough to be classified as active, so recency provides no discriminating power whatsoever.

This feature will not cause the clustering model to fail, but it contributes zero information to outlet differentiation. It should either be re-engineered (e.g., using a continuous days-since-last-order measure rather than a binary active/inactive classification) or dropped before Module 6 runs.

**So what:** If recency is left in the clustering model as-is, it is dead weight — it will not help differentiate outlet segments, and resolving this before clustering will improve the quality of the segments you get out.

---

### 6. The Three Things You Should Act On First

> Based on the full analysis, these are the highest-priority business decisions:

1. **Build a dedicated account management programme for your 4,534 power outlets immediately.**
   These outlets represent 6.79% of your active base but **72.22% of revenue**. They are not adequately served by the same sales motions used for mainstream outlets. A churn event among this group has catastrophic revenue implications — each power outlet is worth, on average, roughly 27× the revenue of a typical outlet. Assign dedicated relationship managers, set churn early-warning alerts, and treat retention of this group as a board-level priority.

2. **Commission a structured win-back analysis on the 49,447 churned outlets before investing in new customer acquisition.**
   You have 49,447 outlets that previously bought from you and stopped. The cost of reactivating a lapsed customer is typically a fraction of acquiring a new one. Segment the churned base by their last-known behavioural profile (using the same 8 features) to identify which churned outlets most resemble your currently active mainstream accounts — those are your highest-probability win-back targets.

3. **Fix the recency feature before running the outlet clustering in Module 6.**
   The recency dimension currently has zero variance across all 62,202 outlets in the clustering population, meaning it contributes nothing to segment differentiation. Re-engineer it using a continuous recency measure (e.g., days since last invoice) rather than the current binary active/inactive flag. A working recency dimension will help the clustering model separate recently-reactivated outlets from consistently loyal ones — a distinction that matters for targeting.

---
*Analysis covers 116,183 outlets (66,736 active, 49,447 churned) across 7,319,529 monthly activity records. Final clustering-ready feature matrix contains 62,202 outlets × 8 features, saved to data/Intermediate/clustering/outlet_features_scaled.parquet.*
